# DeepFALCON GSoC 2026 – Specific Task 2
## Denoising Diffusion Probabilistic Model (DDPM) for Jet Image Simulation

**Model:** DDPM with U-Net backbone + sinusoidal time embeddings  
**Dataset:** Quark/Gluon jet images (3 × 125 × 125)  
**Reference:** Ho et al. *Denoising Diffusion Probabilistic Models* (2020)

### How it works
```
Forward:  x_0 → x_1 → x_2 → ... → x_T  (add noise gradually)
Reverse:  x_T → x_{T-1} → ... → x_0    (U-Net learns to denoise)
```
Reconstruction: corrupt a real jet to timestep t_noise, then denoise back → compare to original

In [3]:
import os, random, time, math
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import h5py
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# ── Config ──────────────────────────────────────────
DATA_PATH   = 'quark-gluon_data-set_n139306.hdf5'  # ← update
T_STEPS     = 1000      # diffusion timesteps
T_RECON     = 500       # noise level for reconstruction
BASE_CH     = 32        # 32 for CPU, 64 for GPU
BATCH_SIZE  = 16        # keep small on CPU
N_EPOCHS    = 2
LR          = 2e-4
MAX_SAMPLES = 1000      # set None for full dataset
OUT_DIR     = 'outputs_diff'
os.makedirs(OUT_DIR, exist_ok=True)

CHANNEL_NAMES = ['ECAL', 'HCAL', 'Tracks']
CHANNEL_CMAPS = ['inferno', 'plasma', 'viridis']

Device: cpu


## 1. Dataset

In [18]:
class JetImageDataset(Dataset):
    def __init__(self, filepath, max_samples=None):
        with h5py.File(filepath, 'r') as f:
            keys = list(f.keys())
            print('HDF5 keys:', keys)
            self.x_key = 'X_jets' if 'X_jets' in keys else ('X' if 'X' in keys else 'jetImage')
            self.y_key = 'y' if 'y' in keys else 'jetLabel'
            total = f[self.x_key].shape[0]
            self.n = min(max_samples, total) if max_samples else total
            # Load only labels (tiny — just integers)
            self.labels = torch.tensor(f[self.y_key][:self.n], dtype=torch.long)
            # Compute per-channel 99th percentile on a small sample only
            sample = f[self.x_key][:min(2000, self.n)]
            if sample.ndim == 4 and sample.shape[-1] == 3:
                sample = sample.transpose(0, 3, 1, 2)
            sample = np.log1p(sample.astype(np.float32))
            self.p99 = [float(np.percentile(sample[:, c], 99)) for c in range(3)]
        self.filepath = filepath
        print(f'Dataset ready: {self.n} events (lazy HDF5 loading)')

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        # Open file per-sample — avoids loading everything into RAM
        with h5py.File(self.filepath, 'r') as f:
            x = f[self.x_key][idx]                     # loads ONE event
        if x.ndim == 3 and x.shape[-1] == 3:
            x = x.transpose(2, 0, 1)                   # HWC → CHW
        x = np.log1p(x.astype(np.float32))
        for c in range(3):
            if self.p99[c] > 0:
                x[c] = np.clip(x[c] / self.p99[c], 0, 1)
        x = x * 2.0 - 1.0                             # → [-1, 1]
        return torch.tensor(x, dtype=torch.float32), self.labels[idx]

# class JetImageDataset(Dataset):
#     def __init__(self, filepath, max_samples=None):
#         with h5py.File(filepath, 'r') as f:
#             X = f['X_jets'][:]
#             self.labels = torch.tensor(f['y'][:], dtype=torch.long)
#         # X shape: (N, 125, 125, 3) → (N, 3, 125, 125)
#         if X.ndim == 4 and X.shape[-1] == 3:
#             X = np.transpose(X, (0, 3, 1, 2))
#         X = X.astype(np.float32)
#         X = np.log1p(X)
#         for c in range(3):
#             p = np.percentile(X[:, c], 99)
#             if p > 0:
#                 X[:, c] = np.clip(X[:, c] / p, 0, 1)
#         if max_samples:
#             X = X[:max_samples]
#             self.labels = self.labels[:max_samples]
#         X = X * 2.0 - 1.0  # scale to [-1, 1] for diffusion
#         self.data = torch.tensor(X, dtype=torch.float32)
#         print(f'Loaded {len(self.data)} events')
#     def __len__(self):
#         return len(self.data)
#     def __getitem__(self, idx):
#         return self.data[idx], self.labels[idx]

full_ds = JetImageDataset(DATA_PATH, max_samples=MAX_SAMPLES)
n_tr = int(.8 * len(full_ds))
n_vl = int(.1 * len(full_ds))
n_te = len(full_ds) - n_tr - n_vl
train_ds, val_ds, test_ds = random_split(full_ds, [n_tr, n_vl, n_te],
    generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train/Val/Test: {n_tr}/{n_vl}/{n_te}')


HDF5 keys: ['X_jets', 'm0', 'pt', 'y']
Dataset ready: 1000 events (lazy HDF5 loading)
Train/Val/Test: 800/100/100


## 2. DDPM Noise Scheduler

In [20]:
class DDPMScheduler:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.T=T; self.device=device
        self.betas     = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas    = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.alpha_bar_prev = F.pad(self.alpha_bar[:-1],(1,0),value=1.0)
        self.sqrt_ab   = self.alpha_bar.sqrt()
        self.sqrt_1mab = (1-self.alpha_bar).sqrt()
        self.sqrt_recip_alpha = (1.0/self.alphas).sqrt()
        self.posterior_var = self.betas*(1-self.alpha_bar_prev)/(1-self.alpha_bar)

    def q_sample(self, x0, t, noise=None):
        if noise is None: noise=torch.randn_like(x0)
        a=self.sqrt_ab[t].view(-1,1,1,1); b=self.sqrt_1mab[t].view(-1,1,1,1)
        return a*x0 + b*noise, noise

    @torch.no_grad()
    def p_sample(self, model, x_t, t_scalar):
        t_b=torch.full((x_t.shape[0],),t_scalar,device=self.device,dtype=torch.long)
        eps=model(x_t,t_b)
        mean=self.sqrt_recip_alpha[t_scalar]*(x_t-self.betas[t_scalar]/self.sqrt_1mab[t_scalar]*eps)
        if t_scalar==0: return mean
        return mean + self.posterior_var[t_scalar].sqrt()*torch.randn_like(x_t)

    @torch.no_grad()
    def reconstruct(self, model, x0, t_noise=500):
        t_b=torch.full((x0.shape[0],),t_noise,device=self.device,dtype=torch.long)
        x,_=self.q_sample(x0,t_b)
        for t in tqdm(range(t_noise-1,-1,-1), desc='Denoising', leave=False):
            x=self.p_sample(model,x,t)
        return x

    @torch.no_grad()
    def sample(self, model, n):
        x=torch.randn(n,3,125,125,device=self.device)
        for t in tqdm(range(self.T-1,-1,-1), desc='Generating', leave=False):
            x=self.p_sample(model,x,t)
        return x

scheduler = DDPMScheduler(T=T_STEPS, device=DEVICE)
print('Scheduler ready.')

Scheduler ready.


## 3. Visualise Forward Diffusion Process

In [ ]:
def to01(x): return (x.clamp(-1,1)+1)/2

sample_img = full_ds[0][0].to(DEVICE)
ts = [0, 100, 250, 500, 750, 999]
fig,axes=plt.subplots(3,len(ts),figsize=(len(ts)*2.5,9),facecolor='#0d0d0d')
for col,tv in enumerate(ts):
    tb=torch.tensor([tv],device=DEVICE)
    xn,_=scheduler.q_sample(sample_img.unsqueeze(0),tb)
    img=to01(xn[0]).cpu().numpy()
    for ch in range(3):
        ax=axes[ch,col]
        ax.imshow(img[ch],cmap=CHANNEL_CMAPS[ch],interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        if ch==0: ax.set_title(f't={tv}',color='white',fontsize=9)
        if col==0: ax.set_ylabel(CHANNEL_NAMES[ch],color='white',fontsize=8)
fig.suptitle('Forward Diffusion: Adding Noise',color='white',fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/noise_levels.png',dpi=150,
    bbox_inches='tight',facecolor='#0d0d0d'); plt.show()

## 4. U-Net Model

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim=dim
    def forward(self, t):
        half=self.dim//2
        freqs=torch.exp(-math.log(10000)*torch.arange(half,device=t.device)/(half-1))
        args=t[:,None].float()*freqs[None]
        return torch.cat([args.sin(),args.cos()],dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, tdim):
        super().__init__()
        self.c1=nn.Sequential(nn.GroupNorm(8,in_ch),nn.SiLU(),nn.Conv2d(in_ch,out_ch,3,padding=1))
        self.te=nn.Sequential(nn.SiLU(),nn.Linear(tdim,out_ch))
        self.c2=nn.Sequential(nn.GroupNorm(8,out_ch),nn.SiLU(),nn.Conv2d(out_ch,out_ch,3,padding=1))
        self.sk=nn.Conv2d(in_ch,out_ch,1) if in_ch!=out_ch else nn.Identity()
    def forward(self,x,te): return self.c2(self.c1(x)+self.te(te)[:,:,None,None])+self.sk(x)

class AttentionBlock(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.norm=nn.GroupNorm(8,ch)
        self.attn=nn.MultiheadAttention(ch,num_heads=4,batch_first=True)
    def forward(self,x):
        B,C,H,W=x.shape; h=self.norm(x).view(B,C,H*W).transpose(1,2)
        h,_=self.attn(h,h,h); return x+h.transpose(1,2).view(B,C,H,W)

class UNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=32, time_dim=128):
        super().__init__()
        B=base_ch
        self.te=nn.Sequential(SinusoidalTimeEmbedding(B),nn.Linear(B,time_dim),nn.SiLU(),nn.Linear(time_dim,time_dim))
        self.ic=nn.Conv2d(in_ch,B,3,padding=1)
        self.e1a=ResBlock(B,B,time_dim);   self.e1b=ResBlock(B,B,time_dim)
        self.d1 =nn.Conv2d(B,B*2,4,stride=2,padding=1)
        self.e2a=ResBlock(B*2,B*2,time_dim); self.e2b=ResBlock(B*2,B*2,time_dim)
        self.d2 =nn.Conv2d(B*2,B*4,4,stride=2,padding=1)
        self.e3a=ResBlock(B*4,B*4,time_dim); self.e3b=ResBlock(B*4,B*4,time_dim)
        self.d3 =nn.Conv2d(B*4,B*8,4,stride=2,padding=1)
        self.m1 =ResBlock(B*8,B*8,time_dim); self.ma=AttentionBlock(B*8); self.m2=ResBlock(B*8,B*8,time_dim)
        self.u3 =nn.ConvTranspose2d(B*8,B*4,4,stride=2,padding=1)
        self.r3a=ResBlock(B*8,B*4,time_dim); self.r3b=ResBlock(B*4,B*4,time_dim)
        self.u2 =nn.ConvTranspose2d(B*4,B*2,4,stride=2,padding=1)
        self.r2a=ResBlock(B*4,B*2,time_dim); self.r2b=ResBlock(B*2,B*2,time_dim)
        self.u1 =nn.ConvTranspose2d(B*2,B,  4,stride=2,padding=1)
        self.r1a=ResBlock(B*2,B,time_dim);   self.r1b=ResBlock(B,B,time_dim)
        self.oc =nn.Sequential(nn.GroupNorm(8,B),nn.SiLU(),nn.Conv2d(B,in_ch,1))

    def forward(self,x,t):
        te=self.te(t); x=self.ic(x)
        e1=self.e1b(self.e1a(x,te),te)
        e2=self.e2b(self.e2a(self.d1(e1),te),te)
        e3=self.e3b(self.e3a(self.d2(e2),te),te)
        b=self.m2(self.ma(self.m1(self.d3(e3),te)),te)
        d=F.interpolate(self.u3(b),size=e3.shape[2:],mode='nearest')
        d=self.r3b(self.r3a(torch.cat([d,e3],1),te),te)
        d=F.interpolate(self.u2(d),size=e2.shape[2:],mode='nearest')
        d=self.r2b(self.r2a(torch.cat([d,e2],1),te),te)
        d=F.interpolate(self.u1(d),size=e1.shape[2:],mode='nearest')
        d=self.r1b(self.r1a(torch.cat([d,e1],1),te),te)
        return self.oc(d)

model=UNet(in_ch=3,base_ch=BASE_CH,time_dim=BASE_CH*4).to(DEVICE)
print(f'U-Net params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 5. Training

In [ ]:
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=N_EPOCHS,eta_min=1e-5)
history={'train':[],'val':[]}; best_val=float('inf')

for epoch in range(1,N_EPOCHS+1):
    model.train(); tl=0
    for x0,_ in tqdm(train_loader,desc=f'Ep {epoch}/{N_EPOCHS}',leave=False):
        x0=x0.to(DEVICE)
        t=torch.randint(0,T_STEPS,(x0.shape[0],),device=DEVICE)
        xn,noise=scheduler.q_sample(x0,t)
        loss=F.mse_loss(model(xn,t),noise)
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        tl+=loss.item()
    tl/=len(train_loader)

    model.eval(); vl=0
    with torch.no_grad():
        for x0,_ in val_loader:
            x0=x0.to(DEVICE); t=torch.randint(0,T_STEPS,(x0.shape[0],),device=DEVICE)
            xn,noise=scheduler.q_sample(x0,t)
            vl+=F.mse_loss(model(xn,t),noise).item()
    vl/=len(val_loader)
    sched.step()
    history['train'].append(tl); history['val'].append(vl)
    if vl<best_val: best_val=vl; torch.save(model.state_dict(),f'{OUT_DIR}/ddpm_best.pt')
    if epoch%5==0 or epoch==1:
        print(f'Ep {epoch:3d} | train {tl:.5f} | val {vl:.5f}')

model.load_state_dict(torch.load(f'{OUT_DIR}/ddpm_best.pt',map_location=DEVICE))
print(f'Done. Best val: {best_val:.5f}')

## 6. Loss Curves

In [ ]:
fig,ax=plt.subplots(figsize=(8,4),facecolor='#0d0d0d'); ax.set_facecolor('#1a1a2e')
epochs=range(1,len(history['train'])+1)
ax.plot(epochs,history['train'],color='#00e5ff',lw=1.8,label='Train')
ax.plot(epochs,history['val'],  color='#ff6e40',lw=1.8,ls='--',label='Val')
ax.set_title('DDPM – Noise Prediction MSE',color='white'); ax.tick_params(colors='white')
ax.set_xlabel('Epoch',color='#aaa'); ax.legend(facecolor='#1a1a2e',labelcolor='white')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/loss_curves.png',dpi=150,
    bbox_inches='tight',facecolor='#0d0d0d'); plt.show()

## 7. ★ Reconstruction: Original vs DDPM ★

In [ ]:
# Grab test samples
test_imgs=[]
for x,_ in test_loader:
    test_imgs.append(x)
    if sum(len(t) for t in test_imgs)>=24: break
test_imgs=torch.cat(test_imgs)[:24].to(DEVICE)

print(f'Reconstructing with t_noise={T_RECON}...')
model.eval()
recons = scheduler.reconstruct(model, test_imgs, t_noise=T_RECON)

def to01(x): return (x.clamp(-1,1)+1)/2

n=6; o=to01(test_imgs[:n]).cpu().numpy(); r=to01(recons[:n]).cpu().numpy()
fig,axes=plt.subplots(n,6,figsize=(16,n*2.6),facecolor='#0d0d0d')
for row in range(n):
    for ch in range(3):
        vmax=max(o[row,ch].max(),r[row,ch].max(),1e-6)
        for j,(data,suffix) in enumerate([(o,'Orig.'),(r,'Recon.')]):
            ax=axes[row,ch*2+j]
            ax.imshow(data[row,ch],cmap=CHANNEL_CMAPS[ch],vmin=0,vmax=vmax,interpolation='nearest')
            ax.set_xticks([]); ax.set_yticks([])
            if row==0: ax.set_title(f'{CHANNEL_NAMES[ch]}\n{suffix}',color='white',fontsize=9)
        if ch==0: axes[row,0].set_ylabel(f'Event {row+1}',color='white',fontsize=8)
fig.suptitle(f'DDPM – Original vs Reconstructed (t_noise={T_RECON})',color='white',fontsize=13,y=1.01)
plt.tight_layout(pad=0.4)
plt.savefig(f'{OUT_DIR}/recon_comparison.png',dpi=150,bbox_inches='tight',facecolor='#0d0d0d'); plt.show()

## 8. Evaluation Metrics + Comparison with VAE

In [ ]:
def compute_metrics(orig, recon):
    o=to01(orig).cpu().numpy(); r=to01(recon).cpu().numpy()
    mse=float(np.mean((o-r)**2)); mae=float(np.mean(np.abs(o-r)))
    psnr=float(10*np.log10(1.0/(mse+1e-8)))
    w1=[]; bins=np.linspace(0,1,100)
    for c in range(3):
        ho,_=np.histogram(o[:,c].flatten(),bins=bins,density=True)
        hr,_=np.histogram(r[:,c].flatten(),bins=bins,density=True)
        w1.append(float(np.mean(np.abs(np.cumsum(ho)-np.cumsum(hr)))/len(bins)))
    return {'MSE':mse,'MAE':mae,'PSNR':psnr,
            'W1_ECAL':w1[0],'W1_HCAL':w1[1],'W1_Tracks':w1[2],'W1_mean':np.mean(w1)}

ddpm_m = compute_metrics(test_imgs, recons)
print('── DDPM Metrics ──────────────────')
for k,v in ddpm_m.items(): print(f'  {k:12s}: {v:.6f}')

# ── Paste your VAE metrics here to compare ──
# vae_m = {'MSE': X, 'MAE': X, 'PSNR': X, 'W1_ECAL': X, 'W1_HCAL': X, 'W1_Tracks': X}
# print('\n── VAE Metrics ───────────────────')
# for k,v in vae_m.items(): print(f'  {k:12s}: {v:.6f}')

## 9. Pixel Histograms

In [ ]:
o=to01(test_imgs).cpu().numpy(); r=to01(recons).cpu().numpy()
fig,axes=plt.subplots(1,3,figsize=(13,4),facecolor='#0d0d0d')
bins=np.linspace(0,1,80)
for ch,ax in enumerate(axes):
    ax.set_facecolor('#1a1a2e'); ax.tick_params(colors='white')
    ax.hist(o[:,ch].flatten(),bins=bins,density=True,color='#00e5ff',alpha=0.6,
            label='Original',histtype='stepfilled')
    ax.hist(r[:,ch].flatten(),bins=bins,density=True,color='#ff6e40',alpha=0.6,
            label='DDPM Recon.',histtype='stepfilled')
    ax.set_title(CHANNEL_NAMES[ch],color='white'); ax.set_xlabel('Pixel value',color='#aaa')
    ax.legend(facecolor='#1a1a2e',labelcolor='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#333355')
fig.suptitle('Pixel Distributions',color='white',fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/pixel_histograms.png',dpi=150,
    bbox_inches='tight',facecolor='#0d0d0d'); plt.show()

## 10. Generate New Jet Samples from Pure Noise

In [ ]:
print('Generating 6 new jets from noise (this takes a while on CPU)...')
model.eval()
gen = scheduler.sample(model, n=6)
imgs=to01(gen).cpu().numpy()
fig,axes=plt.subplots(6,3,figsize=(9,16),facecolor='#0d0d0d')
for row in range(6):
    for ch in range(3):
        ax=axes[row,ch]
        ax.imshow(imgs[row,ch],cmap=CHANNEL_CMAPS[ch],interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        if row==0: ax.set_title(CHANNEL_NAMES[ch],color='white')
        if ch==0: ax.set_ylabel(f'Sample {row+1}',color='white',fontsize=8)
fig.suptitle('DDPM Generated Jet Images',color='white',fontsize=12)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/generated.png',dpi=150,
    bbox_inches='tight',facecolor='#0d0d0d'); plt.show()
print('All outputs saved to:', OUT_DIR)